# Spam Classification using AdaBoost and Naive Bayes
This assignment guides you through building a spam classifier using AdaBoost and Naive Bayes.
You will write pseudocode, preprocess data, train models, evaluate accuracy, and compare results.

## 1. Assignment Objectives
- Load and preprocess text data
- Implement pseudocode for both AdaBoost and Naive Bayes
- Train models
- Evaluate performance
- Compare results

## 2. Pseudocode: Naive Bayes Classifier
```
START
INPUT: Training text data with labels
PREPROCESS: Clean text → tokenize → remove stopwords → convert to vectors
CALCULATE prior probabilities for each class
FOR each word in vocabulary:
    CALCULATE likelihood P(word | class)
STORE probabilities
DURING prediction:
    For each class:
        Compute log probability of text belonging to class
    SELECT class with highest probability
END
```

## 3. Pseudocode: AdaBoost Classifier
```
START
INPUT: Preprocessed feature vectors
INITIALIZE: Equal weights for all samples
FOR t = 1 to T (number of weak learners):
    Train weak learner (e.g., decision stump)
    Compute error
    Compute alpha (learner weight)
    UPDATE sample weights
END FOR
FINAL prediction = weighted sum of weak learners
RETURN predicted class
END
```

# Dataset
The Dataset I used https://www.kaggle.com/datasets/venky73/spam-mails-dataset

In [1]:
# 4. Import Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

In [3]:
# 5. Load Sample Dataset (Replace with your own CSV)
data = pd.read_csv("C:\\Users\\rache\\Downloads\\spam_ham_dataset.csv\\spam_ham_dataset.csv")
data.head()

,Unnamed: 0,label,text,label_num
0,605,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
1,2349,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,3624,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,4685,spam,"Subject: photoshop , windows , office . cheap ...",1
4,2030,ham,Subject: re : indian springs\r\nthis deal is t...,0


In [5]:
#Shape of the Data Set
data.shape

(5171, 4)

In [7]:
# Drop the unwanted columns
df =  data.drop(['Unnamed: 0', 'label_num'], axis = 1)

In [9]:
df.head(10)

,label,text
0,ham,Subject: enron methanol ; meter # : 988291\r\n...
1,ham,"Subject: hpl nom for january 9 , 2001\r\n( see..."
2,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar..."
3,spam,"Subject: photoshop , windows , office . cheap ..."
4,ham,Subject: re : indian springs\r\nthis deal is t...
5,ham,Subject: ehronline web address change\r\nthis ...
6,ham,Subject: spring savings certificate - take 30 ...
7,spam,Subject: looking for medication ? we ` re the ...
8,ham,Subject: noms / actual flow for 2 / 26\r\nwe a...
9,ham,"Subject: nominations for oct . 21 - 23 , 2000\..."


In [11]:
df['label'].unique()

array(['ham', 'spam'], dtype=object)

In [13]:
# Value_counts of the Label column
df['label'].value_counts()

label
ham     3672
spam    1499
Name: count, dtype: int64

In [17]:
# Checking the null values
df.isnull().sum()

label    0
text     0
dtype: int64

## TEXT PREPROCESSING
Since the Dataset is Unclean, let's Text Preprocess the dataset with Lowercasing,  removing urls, punctuations, extra spaces, hashtags etc. Removing stopwords, tokenizations and lemmatize the dataset for training the model.

In [20]:
import re

In [22]:
def clean_text(text):
    text = text.lower()                           # lowercase
    text = re.sub(r"http\S+|www\S+", "", text)    # remove URLs
    text = re.sub(r"@\w+|#\w+", "", text)         # remove mentions/hashtags
    text = re.sub(r"[^a-z\s]", "", text)          # remove punctuation/numbers
    text = re.sub(r"\s+", " ", text).strip()      # remove extra spaces
    return text

df["clean_text"] = df["text"].apply(clean_text)

In [24]:
df.head()

,label,text,clean_text
0,ham,Subject: enron methanol ; meter # : 988291\r\n...,subject enron methanol meter this is a follow ...
1,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",subject hpl nom for january see attached file ...
2,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",subject neon retreat ho ho ho we re around to ...
3,spam,"Subject: photoshop , windows , office . cheap ...",subject photoshop windows office cheap main tr...
4,ham,Subject: re : indian springs\r\nthis deal is t...,subject re indian springs this deal is to book...


In [26]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [28]:
import nltk

In [30]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def tokenize_and_lemmatize(text):
    tokens = nltk.word_tokenize(text)  # tokenize
    tokens = [t for t in tokens if t not in stop_words]  # remove stopwords
    tokens = [lemmatizer.lemmatize(t) for t in tokens]   # lemmatization
    return tokens

df["tokens"] = df["clean_text"].apply(tokenize_and_lemmatize)

In [32]:
# Creating the column for vectorization that combines the tokens 
df["text_for_vector"] = df["tokens"].apply(lambda x: " ".join(x))

In [34]:
df.head()

,label,text,clean_text,tokens,text_for_vector
0,ham,Subject: enron methanol ; meter # : 988291\r\n...,subject enron methanol meter this is a follow ...,"[subject, enron, methanol, meter, follow, note...",subject enron methanol meter follow note gave ...
1,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",subject hpl nom for january see attached file ...,"[subject, hpl, nom, january, see, attached, fi...",subject hpl nom january see attached file hpln...
2,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",subject neon retreat ho ho ho we re around to ...,"[subject, neon, retreat, ho, ho, ho, around, w...",subject neon retreat ho ho ho around wonderful...
3,spam,"Subject: photoshop , windows , office . cheap ...",subject photoshop windows office cheap main tr...,"[subject, photoshop, window, office, cheap, ma...",subject photoshop window office cheap main tre...
4,ham,Subject: re : indian springs\r\nthis deal is t...,subject re indian springs this deal is to book...,"[subject, indian, spring, deal, book, teco, pv...",subject indian spring deal book teco pvr reven...


In [36]:
# 6. Preprocessing
X = df['text_for_vector']
y = df['label']
vectorizer = TfidfVectorizer()
X_vec = vectorizer.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)

In [38]:
# 7. Train Naive Bayes Classifier
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
print('Naive Bayes Accuracy:', accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.927536231884058
              precision    recall  f1-score   support

         ham       0.91      1.00      0.95       742
        spam       1.00      0.74      0.85       293

    accuracy                           0.93      1035
   macro avg       0.95      0.87      0.90      1035
weighted avg       0.93      0.93      0.92      1035



In [40]:
# 8. Train AdaBoost Classifier
weak_learner = DecisionTreeClassifier(max_depth=1)
ada = AdaBoostClassifier(estimator=weak_learner, n_estimators=50)
ada.fit(X_train, y_train)
y_pred_ada = ada.predict(X_test)
print('AdaBoost Accuracy:', accuracy_score(y_test, y_pred_ada))
print(classification_report(y_test, y_pred_ada))

AdaBoost Accuracy: 0.9217391304347826
              precision    recall  f1-score   support

         ham       0.98      0.91      0.94       742
        spam       0.81      0.95      0.87       293

    accuracy                           0.92      1035
   macro avg       0.89      0.93      0.91      1035
weighted avg       0.93      0.92      0.92      1035



## 9. Conclusion
- Compare Naive Bayes vs AdaBoost performance
- Discuss errors and improvements

* I have used spam_ham_dataset from Kaggle
* Compare to Naive Bayes, Ada boost performance is improved with Precision and Recall Scores.
* Both models performed better with similar Accuracy.
* The performance may vary when other Classification ML algorithms is used.